# 시간대별 원자료 매출 패널 준비와 검증

종로구·마포구 외식업 10종, 2023~2025년 12분기의 최종 패널에 원자료 **6개 시간대 매출 금액**을 연결한다. 분기의 매출을 일별로 나누거나 공연 후 창에 임의 배분하지 않는다. 미제공 금액은 결측으로 유지한다.

성공 기준은 고정 입력의 해시 일치, 151×10×12×6=108,720개 슬롯 보존, 원자료 총매출·기존 야간 매출의 정확한 일치다. 6개 시간대 합계와 총매출 사이에 차이가 있으면 기록하고 값을 바꾸거나 행을 삭제하지 않는다. 이 노트북은 자료 준비만 수행하며 모형을 적합하지 않는다.

## 1. 경로와 원래 시간대 정의
프로젝트 루트를 찾고 출력은 시간대 확장의 별도 폴더로 한정한다. 00~06·06~11·11~14·14~17·17~21·21~24시의 시간폭은 각각 6·5·3·3·4·3시간이다. 시간폭으로 매출을 나누지 않는다.

In [1]:
from pathlib import Path
from datetime import datetime, timezone
import hashlib, json, platform, importlib.metadata
import numpy as np
import pandas as pd
from IPython.display import display, Markdown

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p / 'final/metadata/seoul_analysis_sources.json').exists())
OUT = ROOT / 'data/analysis/final/time_windows'
REPORT = ROOT / 'reports/final/time_windows'
OUT.mkdir(parents=True, exist_ok=True)
REPORT.mkdir(parents=True, exist_ok=True)
KEY = ['area_id', 'industry_id', 'quarter']
BIN_HOURS = {'00_06': 6, '06_11': 5, '11_14': 3,
             '14_17': 3, '17_21': 4, '21_24': 3}
BIN_SOURCE = {b: f"시간대_{b.replace('_', '~')}_매출_금액" for b in BIN_HOURS}
PANEL_PATH = ROOT / 'data/analysis/final/food_industry_panel.csv'
META_PATH = ROOT / 'reports/final/08_food_scope_metadata.json'
MANIFEST_PATH = ROOT / 'final/metadata/seoul_analysis_sources.json'

def sha256(path):
    with path.open('rb') as stream:
        return hashlib.file_digest(stream, 'sha256').hexdigest()

display(pd.DataFrame([{'time_bin': b, '원자료 열': BIN_SOURCE[b],
                      'bin_hours': h} for b, h in BIN_HOURS.items()]))

,time_bin,원자료 열,bin_hours
0,00_06,시간대_00~06_매출_금액,6
1,06_11,시간대_06~11_매출_금액,5
2,11_14,시간대_11~14_매출_금액,3
3,14_17,시간대_14~17_매출_금액,3
4,17_21,시간대_17~21_매출_금액,4
5,21_24,시간대_21~24_매출_금액,3


**해석:** 6개 시간대는 원래 제공된 분기 매출의 구분이다. 긴 형태로 펼친 행 수가 늘어나더라도 독립적인 상권이나 분기 표본이 늘어나는 것은 아니다.

## 2. 고정 입력 해시와 기본 패널 확인
최종 04번이 기록한 패널 해시를 확인한다. 서울시 파일도 공용 최신 경로 대신 검증된 고정 입력 명세의 2023~2025년 매출 CSV만 읽는다.

In [2]:
metadata = json.loads(META_PATH.read_text(encoding='utf-8'))
manifest = json.loads(MANIFEST_PATH.read_text(encoding='utf-8'))
assert manifest['status'] == 'validated' and manifest['validation_passed']
assert sha256(PANEL_PATH) == metadata['panel_sha256'], '최종 패널 해시 불일치'
manifest_expected = next(r['sha256'] for r in metadata['source_files']
                         if r['path'] == MANIFEST_PATH.relative_to(ROOT).as_posix())
assert sha256(MANIFEST_PATH) == manifest_expected, '고정 입력 명세 해시 불일치'
source_records = []
for item in manifest['items']:
    if item['dataset'] == 'sales' and int(item['year']) in [2023, 2024, 2025]:
        entries = [r for r in item['extracted'] if r['path'].lower().endswith('.csv')]
        assert len(entries) == 1
        r = entries[0]
        actual = sha256(ROOT / r['path'])
        assert actual == r['sha256'], f"매출 원본 해시 불일치: {r['path']}"
        source_records.append({'year': int(item['year']), 'path': r['path'],
                               'sha256': actual, 'hash_match': True})
assert sorted(r['year'] for r in source_records) == [2023, 2024, 2025]
base_cols = KEY + ['industry', 'area_name', 'district', 'total_sales',
                   'sales_record_present', 'night_sales']
base = pd.read_csv(PANEL_PATH, dtype={k: str for k in KEY},
                   usecols=base_cols, float_precision='round_trip')
base = base[base_cols].sort_values(KEY).reset_index(drop=True)
assert len(base) == 151 * 10 * 12 and not base.duplicated(KEY).any()
assert base['area_id'].nunique() == 151 and base['industry_id'].nunique() == 10
assert base['quarter'].nunique() == 12 and set(base['district']) == {'종로구', '마포구'}
assert base['sales_record_present'].isin([True, False]).all()
display(pd.DataFrame(source_records)[['year', 'path', 'hash_match']])
display(base.groupby('district').agg(격자행=('quarter', 'size'),
           매출제공행=('total_sales', 'count'), 상권수=('area_id', 'nunique')))

,year,path,hash_match
0,2023,data/raw/seoul_recollection/20261004_091155_b8...,True
1,2024,data/raw/seoul_recollection/20261004_091155_b8...,True
2,2025,data/raw/seoul_recollection/20261004_091155_b8...,True


,격자행,매출제공행,상권수
district,,,
마포구,9240,4326,77
종로구,8880,3335,74


**해석:** 해시 검증은 이전 최종 분석과 같은 입력을 사용한다는 확인이다. 자료 제공의 선택성, 과거 경계·시설 위치의 정확성 또는 인과적 타당성을 보장하지 않는다.

## 3. 원래 6개 금액 열 추출
CSV의 시간대 금액 열을 직접 읽고 대상 상권·업종·분기 키에 연결한다. 원자료에 실제 있는 0원과 자료가 제공되지 않은 결측을 구분한다. 중복 키는 임의 집계하지 않고 중단한다.

In [3]:
rename = {'기준_년분기_코드': 'quarter', '상권_코드': 'area_id',
          '서비스_업종_코드': 'industry_id', '당월_매출_금액': 'raw_total_sales',
          **{column: b for b, column in BIN_SOURCE.items()}}
frames, source_quality = [], []
for record in sorted(source_records, key=lambda r: r['year']):
    raw = pd.read_csv(ROOT / record['path'], encoding='cp949',
                      usecols=list(rename), low_memory=False).rename(columns=rename)
    for k in KEY:
        raw[k] = raw[k].astype('string').str.replace(r'\.0$', '', regex=True)
    assert raw[KEY].notna().all().all() and not raw.duplicated(KEY).any()
    assert set(raw['quarter']) == {f"{record['year']}{q}" for q in range(1, 5)}
    selected = raw.merge(base[KEY], on=KEY, how='inner', validate='one_to_one')
    for c in ['raw_total_sales', *BIN_HOURS]:
        selected[c] = pd.to_numeric(selected[c], errors='raise').astype('Int64')
    frames.append(selected)
    source_quality.append({'year': record['year'], 'source_rows': len(raw),
                           'selected_rows': len(selected), 'duplicate_keys': 0,
                           'missing_bin_values': int(selected[list(BIN_HOURS)].isna().sum().sum())})
raw_selected = pd.concat(frames, ignore_index=True)
assert not raw_selected.duplicated(KEY).any()
wide = base.merge(raw_selected, on=KEY, how='left', validate='one_to_one', indicator=True)
wide['raw_record_present'] = wide.pop('_merge').eq('both')
assert wide['raw_record_present'].eq(wide['sales_record_present']).all()
display(pd.DataFrame(source_quality))
display(wide[KEY + ['total_sales', *BIN_HOURS]].head(6))

,year,source_rows,selected_rows,duplicate_keys,missing_bin_values
0,2023,88246,2586,0,0
1,2024,87179,2575,0,0
2,2025,85732,2500,0,0


,area_id,industry_id,quarter,total_sales,00_06,06_11,11_14,14_17,17_21,21_24
0,3001494,CS100001,20231,4.643128e+10,948910737,462653015,11615716683,5822913990,20183905820,7397179341
1,3001494,CS100001,20232,4.532269e+10,1105140436,428784363,11872513966,5466936788,19628767065,6820549704
2,3001494,CS100001,20233,4.363515e+10,1103603109,443397155,11576626402,4785557222,19267461509,6458504737
3,3001494,CS100001,20234,5.185611e+10,1563524231,413025934,11986796538,5853584103,23388397469,8650785698
4,3001494,CS100001,20241,4.744416e+10,1561950594,401701147,11701902121,5333693660,20830818677,7614095087
5,3001494,CS100001,20242,4.528307e+10,1404149290,439974201,11865980656,4587984934,19642265467,7342711477


**해석:** 키는 최종 패널의 18,120개 조합을 기준으로 유지한다. 원자료에 없는 매출행은 여섯 시간대 모두 결측으로 남는다.

## 4. 기존 총매출·야간 매출과 6개 금액 합계 대조
기존 총매출과 21~24시 매출은 정확히 일치해야 한다. 시간대 합계는 6개 금액이 모두 있을 때만 계산한다. 합계 차이는 별도 감사표에 남기고 보정·삭제하지 않는다.

In [4]:
def exact_equal(left, right):
    return (left.eq(right).fillna(False) | (left.isna() & right.isna())).astype(bool)

wide['raw_total_matches_panel'] = exact_equal(wide['raw_total_sales'], wide['total_sales'])
wide['night_matches_panel'] = exact_equal(wide['21_24'], wide['night_sales'])
wide['observed_bin_count'] = wide[list(BIN_HOURS)].notna().sum(axis=1)
wide['six_bin_sum'] = wide[list(BIN_HOURS)].sum(axis=1, min_count=6)
wide['six_bin_minus_total'] = wide['six_bin_sum'] - wide['total_sales']
wide['six_bin_sum_differs'] = wide['six_bin_minus_total'].notna() & wide['six_bin_minus_total'].ne(0)
wide['negative_bin_count'] = wide[list(BIN_HOURS)].lt(0).sum(axis=1)
audit_columns = KEY + ['sales_record_present', 'raw_record_present', 'total_sales',
    'raw_total_sales', 'night_sales', '21_24', 'observed_bin_count', 'six_bin_sum',
    'six_bin_minus_total', 'six_bin_sum_differs', 'raw_total_matches_panel',
    'night_matches_panel', 'negative_bin_count']
audit = wide[audit_columns].copy()
audit.to_csv(REPORT / '10_sales_reconciliation.csv', index=False, encoding='utf-8-sig')
flagged = audit.loc[(~audit['raw_total_matches_panel']) | (~audit['night_matches_panel']) |
    audit['six_bin_sum_differs'] | audit['negative_bin_count'].gt(0) |
    (audit['sales_record_present'] & audit['observed_bin_count'].ne(6))].copy()
flagged.to_csv(REPORT / '10_sales_flagged_rows.csv', index=False, encoding='utf-8-sig')
assert wide['raw_total_matches_panel'].all(), '총매출 대조 실패: 감사표 확인'
assert wide['night_matches_panel'].all(), '기존 야간 매출 대조 실패: 감사표 확인'
reconcile_summary = pd.DataFrame([{
    '격자행': len(wide), '6구간모두관측': int(wide['observed_bin_count'].eq(6).sum()),
    '일부구간만관측': int(wide['observed_bin_count'].between(1, 5).sum()),
    '총매출불일치': int((~wide['raw_total_matches_panel']).sum()),
    '야간매출불일치': int((~wide['night_matches_panel']).sum()),
    '6구간합계차이행': int(wide['six_bin_sum_differs'].sum()),
    '합계차이최대절댓값_원': float(wide['six_bin_minus_total'].abs().max()),
    '음수구간수': int(wide['negative_bin_count'].sum())}])
display(reconcile_summary)
display(flagged.head(10))

,격자행,6구간모두관측,일부구간만관측,총매출불일치,야간매출불일치,6구간합계차이행,합계차이최대절댓값_원,음수구간수
0,18120,7661,0,0,0,0,0.0,0


,area_id,industry_id,quarter,sales_record_present,raw_record_present,total_sales,raw_total_sales,night_sales,21_24,observed_bin_count,six_bin_sum,six_bin_minus_total,six_bin_sum_differs,raw_total_matches_panel,night_matches_panel,negative_bin_count


In [5]:
display(Markdown(
    f"**대조 해석:** 총매출·기존 야간 매출의 불일치는 각각 "
    f"{int((~wide['raw_total_matches_panel']).sum()):,}·{int((~wide['night_matches_panel']).sum()):,}행이다. "
    f"6개 시간대 합계와 총매출이 다른 행은 {int(wide['six_bin_sum_differs'].sum()):,}개이며 "
    f"최대 절댓값 차이는 {float(wide['six_bin_minus_total'].abs().max()):,.0f}원이다. "
    '차이가 있는 원자료도 그대로 보존했다. 회귀 표본 선택은 후속 모형 노트북에서 별도로 기록해야 한다.'))

**대조 해석:** 총매출·기존 야간 매출의 불일치는 각각 0·0행이다. 6개 시간대 합계와 총매출이 다른 행은 0개이며 최대 절댓값 차이는 0원이다. 차이가 있는 원자료도 그대로 보존했다. 회귀 표본 선택은 후속 모형 노트북에서 별도로 기록해야 한다.

## 5. 긴 형태로 변환하고 시간대별 관측 범위 확인
각 원래 금액을 해당 시간대에 한 번씩 보존한다. 108,720행은 매출이 있는 행과 없는 슬롯을 모두 포함하며, 분석 가능한 관측 수와 다르다.

In [6]:
long = wide.melt(id_vars=base_cols, value_vars=list(BIN_HOURS),
                 var_name='time_bin', value_name='sales_amount')
long['bin_hours'] = long['time_bin'].map(BIN_HOURS).astype(int)
long['sales_amount'] = long['sales_amount'].astype('Int64')
output_columns = KEY + ['industry', 'area_name', 'district', 'time_bin', 'bin_hours',
                        'sales_amount', 'total_sales', 'sales_record_present', 'night_sales']
long = long[output_columns].sort_values(KEY + ['time_bin']).reset_index(drop=True)
assert len(long) == 108720 and not long.duplicated(KEY + ['time_bin']).any()
assert long.groupby(KEY)['time_bin'].nunique().eq(6).all()
assert long.loc[~long['sales_record_present'], 'sales_amount'].isna().all()
night = long.loc[long['time_bin'].eq('21_24')]
assert exact_equal(night['sales_amount'], night['night_sales']).all()
coverage = long.groupby('time_bin', sort=False).agg(
    expected_rows=('quarter', 'size'), observed_sales_rows=('sales_amount', 'count'),
    zero_sales_rows=('sales_amount', lambda x: int(x.eq(0).sum())),
    negative_sales_rows=('sales_amount', lambda x: int(x.lt(0).sum())),
    bin_hours=('bin_hours', 'first')).reset_index()
coverage['missing_sales_rows'] = coverage['expected_rows'] - coverage['observed_sales_rows']
coverage['observed_pct'] = 100 * coverage['observed_sales_rows'] / coverage['expected_rows']
by_quarter = long.groupby(['quarter', 'time_bin']).agg(
    expected_rows=('industry_id', 'size'), observed_sales_rows=('sales_amount', 'count')).reset_index()
by_industry = long.groupby(['industry_id', 'industry', 'time_bin']).agg(
    expected_rows=('quarter', 'size'), observed_sales_rows=('sales_amount', 'count')).reset_index()
display(coverage)
display(long.head(6))

,time_bin,expected_rows,observed_sales_rows,zero_sales_rows,negative_sales_rows,bin_hours,missing_sales_rows,observed_pct
0,00_06,18120,7661,3019,0,6,10459,42.279249
1,06_11,18120,7661,1670,0,5,10459,42.279249
2,11_14,18120,7661,413,0,3,10459,42.279249
3,14_17,18120,7661,238,0,3,10459,42.279249
4,17_21,18120,7661,53,0,4,10459,42.279249
5,21_24,18120,7661,843,0,3,10459,42.279249


,area_id,industry_id,quarter,industry,area_name,district,time_bin,bin_hours,sales_amount,total_sales,sales_record_present,night_sales
0,3001494,CS100001,20231,한식음식점,종로·청계 관광특구,종로구,00_06,6,948910737,4.643128e+10,True,7.397179e+09
1,3001494,CS100001,20231,한식음식점,종로·청계 관광특구,종로구,06_11,5,462653015,4.643128e+10,True,7.397179e+09
2,3001494,CS100001,20231,한식음식점,종로·청계 관광특구,종로구,11_14,3,11615716683,4.643128e+10,True,7.397179e+09
3,3001494,CS100001,20231,한식음식점,종로·청계 관광특구,종로구,14_17,3,5822913990,4.643128e+10,True,7.397179e+09
4,3001494,CS100001,20231,한식음식점,종로·청계 관광특구,종로구,17_21,4,20183905820,4.643128e+10,True,7.397179e+09
5,3001494,CS100001,20231,한식음식점,종로·청계 관광특구,종로구,21_24,3,7397179341,4.643128e+10,True,7.397179e+09


**해석:** 시간대별 0원은 원자료에 실제 기록된 값이다. 결측을 채워서 만든 0원이 아니다. 후속 분석은 여섯 시간대가 모두 제공된 상권×업종×분기 표본의 수를 별도로 표시하고, 같은 상권의 시간대·업종·분기를 독립 표본으로 세지 않아야 한다.

## 6. 산출물과 재현 메타데이터 저장
새 매출 패널과 감사표만 시간대 확장 폴더에 저장한다. 원자료·기존 패널·기존 분석 결과를 수정하지 않고 네트워크 요청도 하지 않는다.

In [7]:
outputs = {
    OUT / 'sales_time_panel.csv': long,
    REPORT / '10_sales_source_quality.csv': pd.DataFrame(source_quality),
    REPORT / '10_sales_source_fingerprints.csv': pd.DataFrame(source_records),
    REPORT / '10_sales_bin_coverage.csv': coverage,
    REPORT / '10_sales_quarter_bin_coverage.csv': by_quarter,
    REPORT / '10_sales_industry_bin_coverage.csv': by_industry,
    REPORT / '10_sales_reconciliation.csv': audit,
    REPORT / '10_sales_flagged_rows.csv': flagged}
for path, frame in outputs.items():
    frame.to_csv(path, index=False, encoding='utf-8-sig')
checks = [
    ('fixed_panel_sha256', sha256(PANEL_PATH) == metadata['panel_sha256']),
    ('fixed_manifest_sha256', sha256(MANIFEST_PATH) == manifest_expected),
    ('three_raw_sha256', all(sha256(ROOT/r['path']) == r['sha256'] for r in source_records)),
    ('base_keys_unique', not base.duplicated(KEY).any()),
    ('full_151x10x12x6_grid', len(long) == 108720),
    ('long_keys_unique', not long.duplicated(KEY + ['time_bin']).any()),
    ('six_bins_per_base_key', bool(long.groupby(KEY)['time_bin'].nunique().eq(6).all())),
    ('raw_presence_exact', bool(wide['raw_record_present'].eq(wide['sales_record_present']).all())),
    ('total_sales_exact', bool(wide['raw_total_matches_panel'].all())),
    ('night_sales_exact', bool(wide['night_matches_panel'].all())),
    ('missing_sales_not_zero_imputed', bool(long.loc[~long['sales_record_present'], 'sales_amount'].isna().all())),
    ('bin_widths_sum_24', sum(BIN_HOURS.values()) == 24)]
validation = pd.DataFrame(checks, columns=['check', 'passed'])
validation.to_csv(REPORT / '10_sales_validation.csv', index=False, encoding='utf-8-sig')
assert validation['passed'].all()
summary = {
    'created_utc': datetime.now(timezone.utc).isoformat(), 'status': 'completed',
    'scope': '종로구·마포구/외식업10종/2023~2025분기/원자료6시간대',
    'base_rows': len(base), 'expected_long_rows': 108720, 'long_rows': len(long),
    'areas': int(base['area_id'].nunique()), 'industries': int(base['industry_id'].nunique()),
    'quarters': sorted(base['quarter'].unique().tolist()), 'bin_hours': BIN_HOURS,
    'observed_base_rows': int(base['total_sales'].notna().sum()),
    'complete_six_bin_base_rows': int(wide['observed_bin_count'].eq(6).sum()),
    'partial_bin_base_rows': int(wide['observed_bin_count'].between(1,5).sum()),
    'observed_bin_rows': int(long['sales_amount'].notna().sum()),
    'missing_bin_rows': int(long['sales_amount'].isna().sum()),
    'six_bin_sum_difference_rows': int(wide['six_bin_sum_differs'].sum()),
    'six_bin_sum_max_absolute_difference_won': float(wide['six_bin_minus_total'].abs().max()),
    'total_sales_mismatch_rows': int((~wide['raw_total_matches_panel']).sum()),
    'night_sales_mismatch_rows': int((~wide['night_matches_panel']).sum()),
    'negative_bin_rows': int(long['sales_amount'].lt(0).sum()),
    'checks_passed': int(validation['passed'].sum()), 'checks_total': len(validation),
    'network_requests': 0, 'model_fits': 0, 'daily_sales_reconstructed': False,
    'missing_sales_zero_imputed': False, 'sales_divided_by_bin_hours': False,
    'source_manifest': MANIFEST_PATH.relative_to(ROOT).as_posix(),
    'source_manifest_sha256': sha256(MANIFEST_PATH),
    'scope_metadata_sha256': sha256(META_PATH), 'base_panel_sha256': sha256(PANEL_PATH),
    'raw_sources': source_records,
    'output_sha256': {path.relative_to(ROOT).as_posix(): sha256(path) for path in outputs},
    'limitations': ['quarterly_not_daily_or_event_postwindow_sales',
        'unequal_original_time_bin_widths', 'selective_sales_missingness',
        'current_area_boundaries_applied_to_all_quarters',
        'six_bins_are_not_six_independent_clusters', 'no_sales_per_store_interpretation'],
    'python': platform.python_version(),
    'packages': {name: importlib.metadata.version(name) for name in ['pandas','numpy','nbformat']}}
summary['output_sha256'][(REPORT/'10_sales_validation.csv').relative_to(ROOT).as_posix()] = sha256(REPORT/'10_sales_validation.csv')
(REPORT / '10_sales_summary.json').write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding='utf-8')
display(validation)
print(f"저장: {len(long):,}행 / 관측금액 {summary['observed_bin_rows']:,}행 / 통과 {len(validation)}개")

,check,passed
0,fixed_panel_sha256,True
1,fixed_manifest_sha256,True
2,three_raw_sha256,True
3,base_keys_unique,True
4,full_151x10x12x6_grid,True
5,long_keys_unique,True
6,six_bins_per_base_key,True
7,raw_presence_exact,True
8,total_sales_exact,True
9,night_sales_exact,True


저장: 108,720행 / 관측금액 45,966행 / 통과 12개


**후속 사용:** `sales_time_panel.csv`를 상권·분기·시간대별 공연 활동 노출과 연결한다. 매출은 원래 시간대 전체의 분기 금액이며, 공연 종료 후 0~3시간 또는 0~6시간의 매출로 복원된 것이 아니다. 이 노트북은 회귀계수·유의성·인과효과를 산출하지 않는다.